# Echogram Data Dimensions

An analysis of the spatial dimensions across all 89 echogram files in the dataset.
Each file corresponds to one recording segment from the Simrad EK80 echosounder,
and the number of pings (width) and range samples (height) varies between files
depending on recording duration and instrument configuration.

In [ ]:
import sys
sys.path.insert(0, '../../../..')

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

DATASETS = Path('/Users/axelhagen/Master/Datasets')
ZARR_DIR = DATASETS / 'echopype_zarr'
PATCH_CACHE = DATASETS / 'patch_cache'

zarr_stores = sorted(ZARR_DIR.glob('*.zarr'))
print(f'{len(zarr_stores)} zarr stores')

## Collect dimensions from all zarr stores

Open each zarr store and read its `ping_time` and `range_sample` dimensions, plus the
time span and max depth. The two channels (38 kHz and 200 kHz) share the same ping
axis, so one set of dimensions per file is enough.

In [ ]:
records = []

for zp in zarr_stores:
    try:
        ds = xr.open_zarr(str(zp), consolidated=False)
        n_pings = ds.sizes['ping_time']
        n_range = ds.sizes['range_sample']

        ping_times = ds['ping_time'].values
        t_start = pd.Timestamp(ping_times[0])
        t_end = pd.Timestamp(ping_times[-1])
        duration_min = (t_end - t_start).total_seconds() / 60

        # Compute max depth from echo_range if available
        if 'echo_range' in ds:
            max_depth = float(ds['echo_range'].isel(channel=0, ping_time=0).values[-1])
        else:
            max_depth = np.nan

        # File size on disk
        import subprocess
        result = subprocess.run(['du', '-sk', str(zp)], capture_output=True, text=True)
        size_mb = int(result.stdout.split()[0]) / 1024

        records.append({
            'file': zp.stem,
            'n_pings': n_pings,
            'n_range_samples': n_range,
            'duration_min': round(duration_min, 2),
            'max_depth_m': round(max_depth, 1),
            'size_mb': round(size_mb, 1),
            'start_time': t_start,
        })
        ds.close()
    except Exception as e:
        print(f'  Error reading {zp.name}: {e}')

df = pd.DataFrame(records).sort_values('start_time').reset_index(drop=True)
print(f'Collected dimensions from {len(df)} files')
df[['file', 'n_pings', 'n_range_samples', 'duration_min', 'max_depth_m']].head(10)

In [ ]:
print('=== Echogram dimensions across all files ===\n')
for col, label, unit in [
    ('n_pings', 'Width (pings)', 'pings'),
    ('n_range_samples', 'Height (range samples)', 'samples'),
    ('duration_min', 'Recording duration', 'min'),
    ('max_depth_m', 'Max depth', 'm'),
]:
    vals = df[col].dropna()
    print(f'{label}:')
    print(f'  min={vals.min():.0f}  max={vals.max():.0f}  '
          f'mean={vals.mean():.0f}  median={vals.median():.0f}  std={vals.std():.0f} {unit}')
    print()

print(f'Total pixels: {(df["n_pings"] * df["n_range_samples"]).sum():,.0f}')
print(f'Total data on disk: {df["size_mb"].sum():.0f} MB')

## Dimension distributions

Histograms of width (pings) and height (range samples) across all files. The width
varies much more than the height because it depends on recording duration, while the
height is set by the instrument's configured maximum range.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Width histogram
ax = axes[0]
ax.hist(df['n_pings'], bins=25, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(df['n_pings'].median(), color='firebrick', ls='--', lw=1.5,
           label=f'median = {df["n_pings"].median():.0f}')
ax.set_xlabel('Number of pings (width)')
ax.set_ylabel('Number of files')
ax.set_title('Echogram width distribution')
ax.legend()

# Height histogram
ax = axes[1]
ax.hist(df['n_range_samples'], bins=25, color='darkorange', edgecolor='white', alpha=0.85)
ax.axvline(df['n_range_samples'].median(), color='firebrick', ls='--', lw=1.5,
           label=f'median = {df["n_range_samples"].median():.0f}')
ax.set_xlabel('Number of range samples (height)')
ax.set_ylabel('Number of files')
ax.set_title('Echogram height distribution')
ax.legend()

plt.tight_layout()
plt.show()

## Width vs. height scatter

Each point is one echogram file. Colour encodes recording duration — longer recordings
produce wider echograms. Height clusters tightly because the instrument's range setting
doesn't change much between files.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

sc = ax.scatter(df['n_pings'], df['n_range_samples'],
                c=df['duration_min'], cmap='viridis', s=50, edgecolors='k', linewidths=0.4)
cb = plt.colorbar(sc, ax=ax, label='Recording duration (min)')

ax.set_xlabel('Number of pings (width)')
ax.set_ylabel('Number of range samples (height)')
ax.set_title('Echogram dimensions — each point is one file')

plt.tight_layout()
plt.show()

## Why do ping counts vary?

The EK80 writes a new `.raw` file each time the operator starts/stops recording or
when the file reaches a size threshold. The ping rate is roughly constant (~15-20 Hz),
so the number of pings is proportional to recording duration. The plot below confirms
this linear relationship.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pings vs duration
ax = axes[0]
ax.scatter(df['duration_min'], df['n_pings'], s=40, alpha=0.7, color='steelblue', edgecolors='k', linewidths=0.3)
ax.set_xlabel('Recording duration (min)')
ax.set_ylabel('Number of pings')
ax.set_title('Ping count scales with recording time')

# Annotate ping rate
mask = df['duration_min'] > 0.5
if mask.sum() > 2:
    ping_rate = df.loc[mask, 'n_pings'] / (df.loc[mask, 'duration_min'] * 60)
    avg_rate = ping_rate.mean()
    ax.text(0.05, 0.92, f'avg ping rate ≈ {avg_rate:.1f} Hz',
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.8))

# Range samples vs max depth
ax = axes[1]
ax.scatter(df['max_depth_m'], df['n_range_samples'], s=40, alpha=0.7, color='darkorange', edgecolors='k', linewidths=0.3)
ax.set_xlabel('Max range / depth (m)')
ax.set_ylabel('Number of range samples')
ax.set_title('Range samples scale with configured depth')

plt.tight_layout()
plt.show()

## Per-file overview

A bar chart showing each file's width (pings) and height (range samples) side by side,
sorted chronologically. This gives a visual overview of the full dataset and makes
outliers easy to spot.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

labels = [f.replace('D2024', '') for f in df['file']]
x = np.arange(len(df))

# Width
axes[0].bar(x, df['n_pings'], color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axhline(df['n_pings'].median(), color='firebrick', ls='--', lw=1, alpha=0.7)
axes[0].set_ylabel('Pings (width)')
axes[0].set_title('Per-file echogram dimensions (chronological)')

# Height
axes[1].bar(x, df['n_range_samples'], color='darkorange', edgecolor='white', linewidth=0.3)
axes[1].axhline(df['n_range_samples'].median(), color='firebrick', ls='--', lw=1, alpha=0.7)
axes[1].set_ylabel('Range samples (height)')
axes[1].set_xlabel('File')

axes[1].set_xticks(x[::5])
axes[1].set_xticklabels(labels[::5], rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

## Aspect ratios

Echograms are much wider than they are tall. The aspect ratio (width / height) varies
from about 0.5 to over 5 depending on recording length. This is relevant for the
U-Net training pipeline, which extracts fixed-width patches (256 pings) and resizes
them to square 512 x 512 inputs.

In [ ]:
df['aspect_ratio'] = df['n_pings'] / df['n_range_samples']
df['total_pixels_M'] = (df['n_pings'] * df['n_range_samples']) / 1e6

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Aspect ratio histogram
ax = axes[0]
ax.hist(df['aspect_ratio'], bins=20, color='mediumpurple', edgecolor='white', alpha=0.85)
ax.axvline(1.0, color='gray', ls=':', lw=1, label='square (1:1)')
ax.axvline(df['aspect_ratio'].median(), color='firebrick', ls='--', lw=1.5,
           label=f'median = {df["aspect_ratio"].median():.2f}')
ax.set_xlabel('Aspect ratio (width / height)')
ax.set_ylabel('Number of files')
ax.set_title('Aspect ratio distribution')
ax.legend(fontsize=9)

# Total pixels per file
ax = axes[1]
ax.hist(df['total_pixels_M'], bins=20, color='seagreen', edgecolor='white', alpha=0.85)
ax.axvline(df['total_pixels_M'].median(), color='firebrick', ls='--', lw=1.5,
           label=f'median = {df["total_pixels_M"].median():.1f} M')
ax.set_xlabel('Total pixels (millions)')
ax.set_ylabel('Number of files')
ax.set_title('File size in pixels')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## Patch dimensions

The training pipeline extracts 256-ping-wide patches with a stride of 64. The patch
height equals the echogram height (range samples) and varies between files. These
patches are resized to 512 x 512 before entering the U-Net.

In [ ]:
PATCH_WIDTH = 256
STRIDE = 64

# Compute expected number of patches per file
df['n_patches_expected'] = ((df['n_pings'] - PATCH_WIDTH) // STRIDE + 1).clip(lower=0)

# Load actual patch dimensions from the cached NPZ files
npz_files = sorted(PATCH_CACHE.glob('*.npz'))
patch_heights = []
patches_per_npz = []

for npz_path in npz_files:
    with np.load(npz_path) as data:
        p = data['patches']  # (N, 3, H, W)
        patch_heights.extend([p.shape[2]] * p.shape[0])
        patches_per_npz.append(p.shape[0])

patch_heights = np.array(patch_heights)
print(f'Loaded {len(patch_heights)} patches from {len(npz_files)} NPZ files')
print(f'Patch width: {PATCH_WIDTH} (fixed)')
print(f'Patch heights: min={patch_heights.min()}, max={patch_heights.max()}, '
      f'median={np.median(patch_heights):.0f}, unique values={len(np.unique(patch_heights))}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Patch height distribution
ax = axes[0]
ax.hist(patch_heights, bins=30, color='teal', edgecolor='white', alpha=0.85)
ax.set_xlabel('Patch height (range samples)')
ax.set_ylabel('Number of patches')
ax.set_title('Patch height distribution')

# Patches per file vs echogram width
ax = axes[1]
ax.scatter(df['n_pings'], df['n_patches_expected'], s=40, alpha=0.7,
           color='steelblue', edgecolors='k', linewidths=0.3)
ax.set_xlabel('Echogram width (pings)')
ax.set_ylabel('Expected patches (stride=64)')
ax.set_title('Wider echograms yield more patches')

# Resize factor — how much each patch is stretched/compressed to reach 512x512
unique_heights = np.unique(patch_heights)
resize_factors_h = 512 / unique_heights
resize_factors_w = 512 / PATCH_WIDTH

ax = axes[2]
ax.barh(range(len(unique_heights)), resize_factors_h, color='coral', edgecolor='white')
ax.set_yticks(range(len(unique_heights)))
ax.set_yticklabels(unique_heights, fontsize=8)
ax.set_xlabel('Vertical resize factor (→ 512 px)')
ax.set_ylabel('Original height')
ax.set_title(f'Resize to 512×512 (horiz: {resize_factors_w:.2f}×)')
ax.axvline(1.0, color='gray', ls=':', lw=1)

plt.tight_layout()
plt.show()

## Timeline and recording gaps

The files are from a single survey (31 Jan – 1 Feb 2024). Plotting the start time
and duration of each file reveals the recording pattern and any gaps between files.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Horizontal bars: each file is a time span
for i, row in df.iterrows():
    start_h = (row['start_time'] - df['start_time'].iloc[0]).total_seconds() / 3600
    ax.barh(i, row['duration_min'] / 60, left=start_h, height=0.7,
            color=plt.cm.viridis(row['n_pings'] / df['n_pings'].max()),
            edgecolor='white', linewidth=0.3)

ax.set_xlabel('Hours since first recording')
ax.set_ylabel('File index (chronological)')
ax.set_title('Recording timeline — bar width = file duration, colour = ping count')

sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=plt.Normalize(df['n_pings'].min(), df['n_pings'].max()))
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Number of pings')

plt.tight_layout()
plt.show()

# Compute gaps between consecutive files
gaps = []
for i in range(1, len(df)):
    prev_end = df.iloc[i-1]['start_time'] + pd.Timedelta(minutes=df.iloc[i-1]['duration_min'])
    gap_sec = (df.iloc[i]['start_time'] - prev_end).total_seconds()
    gaps.append(gap_sec)

gaps = np.array(gaps)
print(f'Gaps between consecutive files:')
print(f'  min={gaps.min():.0f}s  max={gaps.max():.0f}s  '
      f'mean={gaps.mean():.0f}s  median={np.median(gaps):.0f}s')
print(f'  {(gaps < 10).sum()} gaps < 10s (near-continuous recording)')
print(f'  {(gaps > 60).sum()} gaps > 1 min')

## Depth configuration changes

The height (range samples) changes when the operator adjusts the instrument's maximum
range setting. The plot below groups files by their height to show how many files
share each configuration.

In [ ]:
height_counts = df.groupby('n_range_samples').agg(
    n_files=('file', 'count'),
    avg_depth=('max_depth_m', 'mean'),
    avg_pings=('n_pings', 'mean'),
).sort_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(height_counts)), height_counts['n_files'],
              color='darkorange', edgecolor='white')
ax.set_xticks(range(len(height_counts)))
ax.set_xticklabels([f'{h}\n({d:.0f} m)' for h, d in
                     zip(height_counts.index, height_counts['avg_depth'])],
                    fontsize=8)
ax.set_xlabel('Range samples (height) — with average max depth')
ax.set_ylabel('Number of files')
ax.set_title('Files per range configuration')

# Annotate bar values
for bar, val in zip(bars, height_counts['n_files']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Combined dimension map

A heatmap-style view showing every file's position in (height, width) space, with
the area of each marker proportional to total pixel count.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sizes = df['total_pixels_M'] / df['total_pixels_M'].max() * 300

sc = ax.scatter(df['n_pings'], df['n_range_samples'],
                s=sizes, c=df['duration_min'], cmap='plasma',
                alpha=0.7, edgecolors='k', linewidths=0.5)
plt.colorbar(sc, ax=ax, label='Duration (min)')

ax.set_xlabel('Number of pings (width)')
ax.set_ylabel('Number of range samples (height)')
ax.set_title('All echograms — marker size ∝ total pixels, colour = duration')

# Add reference lines for patch extraction
ax.axvline(PATCH_WIDTH, color='gray', ls=':', lw=1, alpha=0.5)
ax.text(PATCH_WIDTH + 30, ax.get_ylim()[0] + 50, f'patch width = {PATCH_WIDTH}',
        fontsize=8, color='gray')

plt.tight_layout()
plt.show()

# Final summary
print(f'\n{"="*55}')
print(f'Dataset: {len(df)} echogram files')
print(f'Width:   {df["n_pings"].min()} – {df["n_pings"].max()} pings  '
      f'(median {df["n_pings"].median():.0f})')
print(f'Height:  {df["n_range_samples"].min()} – {df["n_range_samples"].max()} samples  '
      f'(median {df["n_range_samples"].median():.0f})')
print(f'Depth:   {df["max_depth_m"].min():.0f} – {df["max_depth_m"].max():.0f} m')
print(f'Patches: {len(patch_heights)} total  '
      f'(each {PATCH_WIDTH} pings wide, height varies)')
print(f'{"="*55}')